In [ ]:
import os
from dotenv import load_dotenv
from mwrogue.esports_client import EsportsClient
from mwrogue.auth_credentials import AuthCredentials

load_dotenv()

credentials = AuthCredentials(
    username=os.getenv("LEAGUEPEDIA_USER"),
    password=os.getenv("LEAGUEPEDIA_PASS"),
)
site = EsportsClient("lol", credentials=credentials)

In [ ]:
print(load_dotenv())                   # True means the .env file was found
#print(os.getenv("LEAGUEPEDIA_USER"))   # should show YourName@yourbot

info = site.client.api("query", meta="userinfo", uiprop="groups")
print("anon" not in info["query"]["userinfo"])   # True = logged in

In [ ]:
import duckdb
import time

page = "LCK/2026 Season/Rounds 1-2"

con = duckdb.connect("lol_rankings.duckdb")
con.execute(open("docs/raw_schema.sql").read())

In [ ]:
import html

def clean(v):
    return None if v in (None, "") else html.unescape(v)

In [ ]:
def pull_table(cargo_table, raw_table, cols, page, page_col="OverviewPage", allow_empty=False):
    where = f"{page_col} = '{page}'"

    rows = site.cargo_client.query(tables=cargo_table, fields=", ".join(cols), where=where)

    values = [[clean(r.get(c, r.get(c.replace("_", " ")))) for c in cols] for r in rows]

    if not values:
        if allow_empty:
            return 0
        raise ValueError(f"No rows for {cargo_table} / {page}")

    col_names = ", ".join(cols)
    placeholders = ", ".join("?" for _ in cols)

    con.execute("BEGIN")
    con.execute(f"DELETE FROM raw.{raw_table} WHERE {page_col} = ?", [page])
    con.executemany(f"INSERT INTO raw.{raw_table} ({col_names}) VALUES ({placeholders})", values)
    con.execute("INSERT INTO raw.pull_log VALUES (?, ?, now(), ?)", [raw_table, page, len(values)])
    con.execute("COMMIT")

    time.sleep(5)
    return len(values)

In [ ]:
games_cols = ["GameId", "MatchId", "OverviewPage", "Team1", "Team2", "WinTeam", "DateTime_UTC"]
print(pull_table("ScoreboardGames", "scoreboard_games", games_cols, page))

In [ ]:
print(con.execute("""
    SELECT COUNT(*), COUNT(DISTINCT GameId)
    FROM raw.scoreboard_games
    WHERE OverviewPage = ?
""", [page]).fetchone())

In [ ]:
teams_cols = ["GameTeamId", "GameId", "MatchId", "OverviewPage", "Team", "Side",
              "IsWinner", "Dragons", "Barons", "Towers", "Gold", "Kills"]

print(pull_table("ScoreboardTeams", "scoreboard_teams", teams_cols, page))

In [ ]:
print(con.execute("""
    SELECT COUNT(*) FROM raw.scoreboard_teams
    WHERE OverviewPage = ? AND IsWinner
""", [page]).fetchone())

In [ ]:
players_cols = ["GameRoleId", "GameId", "GameTeamId", "MatchId", "OverviewPage", "Link", "Name",
                "Team", "Role", "Side", "Champion", "Kills", "Deaths", "Assists", "Gold", "CS",
                "DamageToChampions", "VisionScore", "DateTime_UTC"]

print(pull_table("ScoreboardPlayers", "scoreboard_players", players_cols, page))

In [ ]:
print(con.execute("""
    SELECT COUNT(*)
    FROM raw.scoreboard_players p
    JOIN raw.scoreboard_teams t USING (GameTeamId)
    JOIN raw.scoreboard_games g ON t.GameId = g.GameId
    WHERE p.OverviewPage = ?
""", [page]).fetchone())

In [ ]:
match_cols = ["MatchId", "OverviewPage", "Team1", "Team2", "Winner", "Team1Score", "Team2Score",
              "FF", "IsNullified", "BestOf", "DateTime_UTC", "Tab", "Patch"]

print(pull_table("MatchSchedule", "match_schedule", match_cols, page))

In [ ]:
print(con.execute("""
    SELECT SUM(Team1Score + Team2Score) FROM raw.match_schedule WHERE OverviewPage = ?
""", [page]).fetchone())

In [ ]:
print(con.execute("SELECT COUNT(*), COUNT(Patch), COUNT(N_GameInMatch) FROM raw.scoreboard_games").fetchone())

In [ ]:
match_game_cols = ["GameId", "MatchId", "OverviewPage", "N_GameInMatch", "Blue", "Red", "Winner",
                   "IsRemake", "IsChronobreak", "FF", "Selection", "PickSelection", "FirstPick",
                   "FirstSelection", "HasSelection", "MVP", "RiotPlatformGameId"]
print(pull_table("MatchScheduleGame", "match_schedule_game", match_game_cols, page))   # 204

In [ ]:
pb_cols = ["GameId", "MatchId", "OverviewPage", "N_GameInMatch", "Team1", "Team2", "Winner", "SwitchedSides"]
for kind in ["Ban", "Pick", "Role"]:
    for team in [1, 2]:
        pb_cols += [f"Team{team}{kind}{i}" for i in range(1, 6)]
pb_cols += ["Team1PicksByRoleOrder", "Team2PicksByRoleOrder", "IsComplete"]
print(pull_table("PicksAndBansS7", "picks_and_bans", pb_cols, page))   # 204

In [ ]:
print(con.execute("""
    SELECT COUNT(*) FROM raw.picks_and_bans p
    JOIN raw.match_schedule_game m USING (GameId)
    WHERE p.Team1 <> m.Blue AND NOT COALESCE(p.SwitchedSides, false)
""").fetchone())

In [ ]:
postgame_cols = ["GameId", "OverviewPage", "RiotPlatformGameId", "StatsPage", "TimelinePage"]
print(pull_table("PostgameJsonMetadata", "postgame_json_metadata", postgame_cols, page))   # 204

In [ ]:
tournament_players_cols = ["OverviewPage", "Team", "Link", "Player", "Role", "N_PlayerInTeam"]
print(pull_table("TournamentPlayers", "tournament_players", tournament_players_cols, page))   # players + coaches

In [ ]:
tournament_cols = ["OverviewPage", "Name", "StandardName", "League", "Region", "Year", "Split",
                   "SplitNumber", "DateStart", "Date", "TournamentLevel", "IsQualifier",
                   "IsPlayoffs", "IsOfficial", "EventType", "Organizer", "Country", "BasePage",
                   "AlternativeNames"]
print(pull_table("Tournaments", "tournaments", tournament_cols, page))   # 1

In [ ]:
results_cols = ["OverviewPage", "RosterPage", "Event", "Tier", "Date", "Team", "Place",
                "Place_Number", "Phase", "Qualified", "Prize_USD", "IsAchievement", "Showmatch"]
print(pull_table("TournamentResults", "tournament_results", results_cols, page, page_col="RosterPage"))

In [ ]:
rows = site.cargo_client.query(tables="IndividualAchievements", fields="OverviewPage, Display",
                               where="OverviewPage LIKE 'LCK/2026%'")
print(rows)

In [ ]:
awards_cols = ["Player", "Link", "Team", "OverviewPage", "Display", "Place", "Place_Number", "AchievementType"]
print(pull_table("IndividualAchievements", "individual_achievements", awards_cols,
                 "PAGE FROM OUTPUT", allow_empty=True))

In [ ]:
con.close()